# train: ResNet-18 transfer learning, 13 classes (M3)

Fine-tunes an ImageNet ResNet-18 on square crops; output index i = `CLASSES[i]`.
torch / torchvision are imported inside functions so this notebook runs without them
(train in Colab with `run_train_cnn.ipynb`).

Experiments (README plan): train three times with different `data.train_sessions`:
sim only, real only, mixed. Same val/test sessions for all.

Milestone: M3 (square classifiers).

**TODO** (see `docs/M3_TODO.md`)
- [ ] `build_model`, `set_backbone_trainable`
- [ ] `train_cnn`: data loaders, optimizer/scheduler, training step (forward, loss, backward,
      step, TensorBoard loss), validation step (accuracy + confusion matrix), checkpoints
- [ ] decide on class-weighted loss (`class_weights`) and a lower backbone lr
- [ ] optional pretraining via `cfg.cnn.pretrain_on` (ChessReD / chesscog synthetic)
- [ ] `CNNClassifier` inference wrapper (the (N, 13) contract for M4)
- [ ] write the self-check (plan at the bottom)

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np

%run ./config.ipynb
%run ./dataset.ipynb

## Model
Replace the final fully connected layer with one of `NUM_CLASSES` outputs. For the first
`freeze_backbone_epochs` epochs only the new head trains, so the random head does not wreck
the pretrained features.

In [ ]:
def build_model(cfg):
    """torchvision ResNet-18 (ImageNet weights if cfg.cnn.pretrained), fc -> NUM_CLASSES."""
    # Design:
    # - Validate cfg.cnn.arch == "resnet18" and cfg.cnn.num_classes == NUM_CLASSES (ValueError).
    # - torchvision.models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 or None).
    # - Replace model.fc with nn.Linear(model.fc.in_features, NUM_CLASSES).
    # TODO: implement
    raise NotImplementedError


def set_backbone_trainable(model, trainable: bool) -> None:
    """Freeze / unfreeze everything except the final fc layer."""
    # Design: loop over named_parameters, set requires_grad for names not starting with "fc.".
    # TODO: implement
    raise NotImplementedError

## Training loop
Writes `<cfg.cnn.out_dir>/<run_name>/{best.pt, last.pt, config.yaml}` and TensorBoard logs
under `<out_dir>/<run_name>/tb`. Returns the path to best.pt.

In [ ]:
def train_cnn(cfg, run_name: str | None = None) -> Path:
    """Fine-tune ResNet-18 on the train sessions, validate on val sessions.

    Writes <cfg.cnn.out_dir>/<run_name>/{best.pt, last.pt, config.yaml} and TensorBoard
    logs under <out_dir>/<run_name>/tb. Returns the path to best.pt.
    Run three times for the experiments: sim only, real only, mixed (different
    train_sessions lists in the config).
    """
    # Design:
    # - Output dir + config.yaml (to_dict(cfg)) first, so every checkpoint is reproducible.
    # - split_by_session -> SquareCropDataset with build_transforms(train=True / False) ->
    #   DataLoader (shuffle only for train).
    # - Loss: CrossEntropyLoss, optionally weighted with class_weights(train df).
    #   Optimizer AdamW(lr=cfg.cnn.lr), CosineAnnealingLR over cfg.cnn.epochs.
    # - Per epoch: set_backbone_trainable(epoch >= freeze_backbone_epochs); train step
    #   (zero_grad, forward, loss, backward, step, log loss to SummaryWriter); val step under
    #   torch.no_grad (accuracy, per-class confusion matrix); scheduler.step().
    # - Checkpoint dict {model state_dict, epoch, val_acc, config}: always last.pt, best.pt when
    #   val accuracy improves.
    # Decisions for you: class weights? lower lr for the backbone (param groups)?
    # pretraining on ChessReD / synthetic first when cfg.cnn.pretrain_on is set?
    # TODO: implement
    raise NotImplementedError

## Inference wrapper
`CNNClassifier.predict_proba(crops) -> (N, 13)` is the model contract M4 consumes
(`docs/contracts.md`).

In [ ]:
class CNNClassifier:
    """Inference wrapper: crops -> (N, 13) probabilities (docs/contracts.md)."""

    def __init__(self, model, cfg, device: str | None = None):
        # Design: pick the device (cuda if available unless given), move the model there in
        # eval mode, keep the eval transform build_transforms(cfg, train=False).
        # TODO: implement
        raise NotImplementedError

    @classmethod
    def from_checkpoint(cls, path: str | Path, cfg=None, device: str | None = None) -> "CNNClassifier":
        """Load a checkpoint written by train_cnn. Uses the config saved in it if cfg is None."""
        # Design: torch.load(map_location="cpu"); config from the checkpoint (_to_ns) if not
        # given; set cnn.pretrained = False (no ImageNet download, weights come from the
        # checkpoint); build_model + load_state_dict.
        # TODO: implement
        raise NotImplementedError

    def predict_proba(self, crops, batch_size: int = 64) -> np.ndarray:
        """crops: list or array of RGB uint8 (H, W, 3). Returns (N, 13) float32, rows sum to 1."""
        # Design: batches under torch.no_grad, transform each crop, softmax over dim 1,
        # concatenate; an empty input returns shape (0, NUM_CLASSES).
        # TODO: implement
        raise NotImplementedError

    def predict(self, crops) -> np.ndarray:
        """argmax class index per crop."""
        # TODO: implement
        raise NotImplementedError

## Self-check
Everything here needs torch, so skip the checks when it is missing.

In [ ]:
# Self-check plan (write these as asserts once the functions above work; wrap them in
# try/except ImportError so the notebook still runs without torch):
# - load_config("configs/train.yaml") with cnn.pretrained = False (no download in a self-check)
#   -> build_model works; after set_backbone_trainable(model, False) only "fc." params
#   have requires_grad.
# - CNNClassifier(model, cfg, device="cpu").predict_proba on two blank (200, 100, 3) crops ->
#   shape (2, 13), rows sum to 1 (atol 1e-5); predict_proba([]) has shape (0, 13).
# - build_model with arch "resnet50" or num_classes 12 raises ValueError.
print("train: self-check not written yet (see plan above)")